# Attention Is All You Need Walkthrough

Goal: connect scaled dot-product attention across formula, tensor shape, local PyTorch code, and the historical Tensor2Tensor implementation.

After this notebook, you should be able to explain why attention scores have shape `(batch, heads, query_tokens, key_tokens)`, why the dot product is divided by `sqrt(d_head)`, and how multi-head outputs are merged back to `d_model`.


## Paper-to-code map

Paper section: scaled dot-product attention and multi-head attention.

Official source to inspect:

- `learning/transformer-deep/official/repos/tensor2tensor/tensor2tensor/models/transformer.py`
- `learning/transformer-deep/official/repos/tensor2tensor/tensor2tensor/layers/common_attention.py`

Runnable local teaching code:

- `learning/transformer-deep/src/mha.py`
- `learning/transformer-deep/src/gpt_mini.py`


In [ ]:
from __future__ import annotations
import importlib.util
import math
import platform
from pathlib import Path
import torch
import torch.nn.functional as F
REPO = Path.cwd()
if not (REPO / "learning").exists():
    REPO = Path.cwd().parent
print("python", platform.python_version())
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("repo", REPO)


## Step 1: tiny Q, K, V tensors

For `B=2`, `H=4`, `T=5`, `D=8`:

```text
q, k, v: (B, H, T, D)
scores:  (B, H, T, T)
out:     (B, H, T, D)
```


In [ ]:
torch.manual_seed(7)
B, H, T, D = 2, 4, 5, 8
q = torch.randn(B, H, T, D)
k = torch.randn(B, H, T, D)
v = torch.randn(B, H, T, D)
scores = q @ k.transpose(-2, -1) / math.sqrt(D)
attn = F.softmax(scores, dim=-1)
out = attn @ v
print("q", tuple(q.shape))
print("scores", tuple(scores.shape))
print("attn row sum", attn[0, 0, 0].sum().item())
print("out", tuple(out.shape))


## Step 2: match the local MHA implementation

The local implementation projects `x` into Q/K/V, applies attention, then applies the output projection.


In [ ]:
mha_path = REPO / "learning" / "transformer-deep" / "src" / "mha.py"
spec = importlib.util.spec_from_file_location("mha_module", mha_path)
mha_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(mha_module)
model = mha_module.MHA(d_model=32, n_head=4, bias=False)
x = torch.randn(2, 6, 32)
y = model(x)
print("input", tuple(x.shape), "output", tuple(y.shape))
assert y.shape == x.shape


## Step 3: official source smoke check

Tensor2Tensor is treated as historical source code. This cell checks files without importing old TensorFlow dependencies.


In [ ]:
official_root = REPO / "learning" / "transformer-deep" / "official" / "repos" / "tensor2tensor"
files = [official_root / "tensor2tensor" / "models" / "transformer.py", official_root / "tensor2tensor" / "layers" / "common_attention.py"]
for file in files:
    print(file.relative_to(REPO), "exists=", file.exists())
assert all(file.exists() for file in files)


## Step 4: scaling ablation

Without `sqrt(d_head)`, logits grow with hidden size and softmax becomes sharper.


In [ ]:
def attention_entropy(d_head: int, scale: bool) -> float:
    torch.manual_seed(0)
    q = torch.randn(1, 1, 1, d_head)
    k = torch.randn(1, 1, 64, d_head)
    logits = q @ k.transpose(-2, -1)
    if scale:
        logits = logits / math.sqrt(d_head)
    probs = logits.softmax(dim=-1)
    return float(-(probs * probs.clamp_min(1e-12).log()).sum())
for d in [8, 32, 128, 512]:
    print({"d_head": d, "scaled": round(attention_entropy(d, True), 3), "unscaled": round(attention_entropy(d, False), 3)})


## Closed-book checks

1. Draw the shape flow from `x` to Q/K/V, scores, attention weights, output, and output projection.
2. Explain why softmax runs over key tokens, not over heads.
3. Explain what can go wrong if the `sqrt(d_head)` scaling is removed.
